# Implement Using Langchain

### Insert Documents 
 - Import the documents here
 - Chunk using the RecursiveTextSplitter
 - Use Document class to create various documents
 - attach meta data to each Document
 - use Pinecone Vector Store to store each document

In [15]:
# Importing important lib

import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from datetime import date
from langchain_pinecone import PineconeVectorStore
from pinecone import ServerlessSpec, Pinecone
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone_text.sparse import BM25Encoder
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import PineconeHybridSearchRetriever
from langchain_google_genai import ChatGoogleGenerativeAI
from yaml import safe_load


In [3]:
def get_chunks():
    docs_path = os.path.join(os.getcwd(), "docs")

    chunks = []
    for doc in os.listdir(docs_path):
        try:
            # Try UTF-8 first, fall back to latin-1 if it fails
            try:
                doc_content = TextLoader(os.path.join(docs_path, doc), encoding="utf-8").load()
            except (UnicodeDecodeError, LookupError):
                print(f"⚠️  UTF-8 failed for {doc}, trying latin-1...")
                doc_content = TextLoader(os.path.join(docs_path, doc), encoding="latin-1").load()

            # chunk the data using MarkdownHeaderTextSplitter
            headers_to_split_on = [
            ("#", "Header 1"),
            ("##", "Header 2"),
            ("###", "Header 3"),
            ("####", "Header 4"),
            ("#####", "Header 5"),
            ("######", "Header 6"),]
            text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
            doc_chunks = text_splitter.split_text(doc_content[0].page_content)
            for doc_chunk in doc_chunks:
                meta_data = {
                    "source": doc,
                    "chunk_content": doc_chunk.page_content,
                    "timestamp": date.today().ctime(),
                    **doc_chunk.metadata
                }
                chunk = Document(page_content=doc_chunk.page_content, metadata=meta_data)
                chunks.append(chunk)
            print(f"✓ Loaded {doc}: {len(doc_chunks)} chunks")
        except Exception as e:
            print(f"❌ Error loading {doc}: {str(e)}")
            continue

    print(f"\n✓ Total chunks loaded: {len(chunks)}")
    return chunks

### initialise the PineconeVectorizer

In [4]:
def get_dense_embedding_model():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

def get_sparse_embedding_model():
    """Create BM25 sparse encoder and fit on document chunks"""
    chunks = get_chunks()
    
    if not chunks:
        raise ValueError("No chunks available for BM25 fitting. Check document loading.")
    
    bm25_encoder = BM25Encoder().default()
    # Extract text content from chunks
    texts = [chunk.page_content for chunk in chunks]
    print(f"Fitting BM25 on {len(texts)} texts...")
    bm25_encoder.fit(texts)
    return bm25_encoder

def initialize_pinecone():
    index_name = "practice-project-company-docs-hybrid"  # Changed name to force new index creation

    pc = Pinecone(api_key=os.getenv('PINECONE_KEY'))

    # Delete old index if it exists to avoid dimension mismatch
    if pc.has_index("practice-project-comapny-docs"):
        print("Deleting old index with incorrect dimension...")
        pc.delete_index("practice-project-comapny-docs")
    
    if not pc.has_index(index_name):
        print(f"Creating new index '{index_name}' with dimension 384...")
        pc.create_index(
            name=index_name,
            dimension=384,
            metric="dotproduct",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )

    index = pc.Index(index_name)

    dense_embeddings = get_dense_embedding_model()
    sparse_embeddings = get_sparse_embedding_model()

    hybrid_retriever = PineconeHybridSearchRetriever(
        index=index,
        embeddings=dense_embeddings,
        sparse_encoder=sparse_embeddings,
    )

    return hybrid_retriever
    # initialize pinecone

### store documents

In [5]:
hybrid_retriever = initialize_pinecone()
hybrid_retriever.add_texts([chunk.page_content for chunk in get_chunks()])

✓ Loaded company_operations.md: 62 chunks
✓ Loaded customer_service_guide.md: 42 chunks
✓ Loaded finance_procedures.md: 63 chunks
✓ Loaded hr_handbook.md: 16 chunks
✓ Loaded it_support_guide.md: 59 chunks
✓ Loaded sample_policy.md: 12 chunks
✓ Loaded technical_documentation.md: 62 chunks

✓ Total chunks loaded: 316
Fitting BM25 on 316 texts...


100%|██████████| 316/316 [00:00<00:00, 449.38it/s]


✓ Loaded company_operations.md: 62 chunks
✓ Loaded customer_service_guide.md: 42 chunks
✓ Loaded finance_procedures.md: 63 chunks
✓ Loaded hr_handbook.md: 16 chunks
✓ Loaded it_support_guide.md: 59 chunks
✓ Loaded sample_policy.md: 12 chunks
✓ Loaded technical_documentation.md: 62 chunks

✓ Total chunks loaded: 316


100%|██████████| 10/10 [00:37<00:00,  3.74s/it]


In [23]:
query = "How many leaves can I take in a calendar year?"
hybrid_retriever.top_k = 6
hybrid_retriever.alpha = 0.7
context = hybrid_retriever.invoke(query)

In [24]:
llm = ChatGoogleGenerativeAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    model="gemini-2.5-flash"
)
llm.invoke('Hello').content

'Hello! How can I help you today?'

In [25]:
def load_chat_prompt_template():
    prompt_path = os.path.join(os.getcwd(), "prompts", "simple_system_prompt.yaml")
    with open(prompt_path, "r") as f:
        loaded_prompt = safe_load(f).get('template')
    return ChatPromptTemplate([
        ("system", loaded_prompt)
    ])

load_chat_prompt_template()

ChatPromptTemplate(input_variables=['sources'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['sources'], input_types={}, partial_variables={}, template='You are an AI assistant which helps answer questions regarding company policy data.\nRules:\n- If You are given a question about company policies you need to strictly adhere to the company\'s policies and guidelines.\n- If a question is asked which is not related to company policies, you should respond with "I am sorry, I can only answer questions related to company policies."\n- Always use the provided context to answer the question. Do not use any information outside of the provided context.\n- If the answer to the question is not found in the provided context, respond with "I am sorry, I do not have enough information to answer that question."\nUSE ONLY THE BELOW CONTEXT TO ANSWER THE QUESTION:\nContext:\n{sources}\n'), additional_kwargs={})])

In [26]:
chat_template = load_chat_prompt_template()
chat_template.extend([
    ('user', "{query}")
])

prompt_value = chat_template.invoke({
    "sources": "\n".join([con.page_content for con in context]),
    "query": query
})
print(prompt_value)

messages=[SystemMessage(content='You are an AI assistant which helps answer questions regarding company policy data.\nRules:\n- If You are given a question about company policies you need to strictly adhere to the company\'s policies and guidelines.\n- If a question is asked which is not related to company policies, you should respond with "I am sorry, I can only answer questions related to company policies."\n- Always use the provided context to answer the question. Do not use any information outside of the provided context.\n- If the answer to the question is not found in the provided context, respond with "I am sorry, I do not have enough information to answer that question."\nUSE ONLY THE BELOW CONTEXT TO ANSWER THE QUESTION:\nContext:\nVacation (PTO): Employees accrue 1.25 days per month (15 days annually) during years 0–2; 1.67 days per month (20 days annually) during years 3–5; and 2.08 days per month (25 days annually) after year 6. Carryover up to 5 days is permitted unless lo

In [27]:
llm.invoke(prompt_value).content

'The number of leaves you can take in a calendar year depends on the type of leave and your years of service for vacation (PTO):\n\n*   **Vacation (PTO):**\n    *   **Years 0–2:** 15 days annually (accrued at 1.25 days per month).\n    *   **Years 3–5:** 20 days annually (accrued at 1.67 days per month).\n    *   **After Year 6:** 25 days annually (accrued at 2.08 days per month).\n    *   You can carry over up to 5 days, unless local law is more generous.\n\n*   **Sick Leave:** 10 paid sick days per year.\n\n*   **Family & Other Leave (event-based):**\n    *   **Parental Leave:** 12 weeks paid for birthing parents; 6 weeks paid for non-birthing parents.\n    *   **Bereavement Leave:** Up to 3 paid days for immediate family; 1 day for extended family.\n    *   **Jury Duty:** Paid in accordance with local law.\n    *   **Military Leave and Civic Duty Leave:** Follow statutory requirements in each jurisdiction.'